In [13]:
"""
Supply/Demand Dashboard - Auto-Update & Deploy
===============================================
Fetches supply/demand data for all regions and generates an interactive dashboard
"""

import requests
import json
import os
import subprocess
from datetime import datetime, timedelta
from collections import defaultdict

# ── CONFIG ────────────────────────────────────────────────────────────────────
USERNAME = "39a9cfa5-18cf-4f0b-b66c-1634b79e906e"
PASSWORD = "cpLScP8pMDGywJks"
AUTH = (USERNAME, PASSWORD)
BASE_URL = "https://api.connect.spglobal.com/cs/v1/pointlogic"

# Local paths
REPO_DIR = r"C:\Users\melaea\LNGFeedgas"  # Files saved here
SUPPLY_DEMAND_REPO = r"C:\Users\melaea\SupplyDemand"  # Separate GitHub repo
DATA_FILE = os.path.join(REPO_DIR, "supply_demand_data.json")
HTML_FILE = os.path.join(REPO_DIR, "supply_demand_dashboard.html")
DEPLOY_HTML = os.path.join(SUPPLY_DEMAND_REPO, "index.html")  # Copy to SupplyDemand repo
DEPLOY_DATA = os.path.join(SUPPLY_DEMAND_REPO, "supply_demand_data.json")

# Fetch last 30 days of data
NUM_DAYS = 30
# ─────────────────────────────────────────────────────────────────────────────

REGIONS = [
    {"id": 26158, "name": "Canada"},
    {"id": 26160, "name": "Mexico"},
    {"id": 26105, "name": "Mid-Continent"},
    {"id": 26104, "name": "Northeast"},
    {"id": 26106, "name": "Rocky Mountain"},
    {"id": 26107, "name": "Southeast"},
    {"id": 26108, "name": "Texas"},
    {"id": 26109, "name": "Western"},
]

REGION_COLORS = {
    "Canada": "#a78bfa",
    "Mexico": "#fb8500",
    "Mid-Continent": "#2ecc71",
    "Northeast": "#00b4d8",
    "Rocky Mountain": "#ffd166",
    "Southeast": "#ff6b35",
    "Texas": "#f72585",
    "Western": "#4cc9f0",
}

# Categorize products into Supply and Demand
SUPPLY_PRODUCTS = [
    "Production",
    "Net Inflows",
    "Net Flows from",  # Prefix for inflows
    "Storage Withdrawals",
    "Sample Storage",  # If positive = withdrawal
]

DEMAND_PRODUCTS = [
    "Res/Com",
    "Industrial",
    "Power",
    "Fuel and Pipe Loss",
    "Net Outflows",
    "Net Flows to",  # Prefix for outflows
]

BALANCE_PRODUCTS = [
    "Implied Storage/Balancing",
    "Total Supply",
    "Total Demand",
    "Total Consumption",
]


def fmt_date(dt):
    """Windows-safe date format"""
    return f"{dt.month}/{dt.day}/{dt.year}"


def fetch_supply_demand_data():
    """Fetch supply/demand data for all regions over the last NUM_DAYS."""
    print(f"📡 Fetching supply/demand data for {len(REGIONS)} regions ({NUM_DAYS} days)...")
    
    today = datetime.today()
    dates = [(today - timedelta(days=i)).strftime("%Y-%m-%d") 
             for i in range(NUM_DAYS - 1, -1, -1)]
    
    all_data = {}
    
    for region in REGIONS:
        rid = region["id"]
        rname = region["name"]
        print(f"\n   {rname} (id={rid})")
        
        all_data[str(rid)] = {}
        
        for date_str in dates:
            # Convert to API format
            dt = datetime.strptime(date_str, "%Y-%m-%d")
            report_date = fmt_date(dt)
            
            url = f"{BASE_URL}/supplyDemand/region/{rid}"
            params = {"reportDate": report_date}
            
            try:
                resp = requests.get(url, auth=AUTH, params=params, timeout=30)
                resp.raise_for_status()
                data = resp.json().get("Data", [])
                
                # Store as dict keyed by product
                products = {}
                for record in data:
                    product = record.get("product")
                    volume = record.get("volume_mmcfd", 0)
                    if product:
                        products[product] = volume
                
                all_data[str(rid)][date_str] = products
                print(f"      {date_str}: {len(products)} products", end="\r")
                
            except Exception as e:
                print(f"      {date_str}: ERROR - {e}")
                all_data[str(rid)][date_str] = {}
        
        print(f"      Done - {len(dates)} days fetched              ")
    
    print(f"\n   ✓ All regions fetched")
    return all_data, dates


def save_data(data):
    """Save data to JSON."""
    # Save to LNGFeedgas folder
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=2)
    print(f"   ✓ Data saved to {DATA_FILE}")
    
    # Also copy to SupplyDemand repo
    print(f"   Creating folder: {SUPPLY_DEMAND_REPO}")
    os.makedirs(SUPPLY_DEMAND_REPO, exist_ok=True)
    
    print(f"   Copying data to: {DEPLOY_DATA}")
    with open(DEPLOY_DATA, "w") as f:
        json.dump(data, f, indent=2)
    print(f"   ✓ Data copied to {DEPLOY_DATA}")


def generate_html(data, dates):
    """Generate the supply/demand dashboard."""
    print("   Generating dashboard HTML...")
    
    updated = datetime.now().strftime("%B %#d, %Y at %I:%M %p")
    
    # Build JS data structure
    # Format: { regionId: { date: { product: volume } } }
    js_data_lines = []
    js_data_lines.append(f"const DATES = {json.dumps(dates)};")
    js_data_lines.append(f"const REGION_NAMES = {json.dumps([r['name'] for r in REGIONS])};")
    js_data_lines.append(f"const REGION_IDS = {json.dumps([r['id'] for r in REGIONS])};")
    js_data_lines.append(f"const REGION_COLORS = {json.dumps(REGION_COLORS)};")
    js_data_lines.append(f"const DATA = {json.dumps(data)};")
    
    JS_DATA = "\n".join(js_data_lines)
    
    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Supply & Demand Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Bebas+Neue&family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;600&display=swap" rel="stylesheet">
<style>
:root {{
  --bg:#0a0e17; --panel:#111827; --border:#1e2d45;
  --accent:#00b4d8; --text:#e2e8f0; --muted:#64748b;
  --supply:#2ecc71; --demand:#ff6b35; --balance:#ffd166;
}}
*{{margin:0;padding:0;box-sizing:border-box;}}
body{{background:var(--bg);color:var(--text);font-family:'IBM Plex Sans',sans-serif;min-height:100vh;
  background-image:radial-gradient(ellipse at 20% 0%,rgba(0,212,255,.06) 0%,transparent 50%),
  radial-gradient(ellipse at 80% 100%,rgba(255,107,53,.05) 0%,transparent 50%);}}
header{{padding:24px 40px 16px;border-bottom:1px solid var(--border);display:flex;align-items:flex-end;gap:20px;}}
.logo-mark{{width:4px;height:44px;background:linear-gradient(180deg,#2ecc71,#ff6b35);border-radius:2px;flex-shrink:0;}}
h1{{font-family:'Bebas Neue',sans-serif;font-size:2.6rem;letter-spacing:.08em;line-height:1;color:#fff;}}
.subtitle{{font-family:'IBM Plex Mono',monospace;font-size:.68rem;color:var(--accent);letter-spacing:.15em;text-transform:uppercase;margin-bottom:3px;}}
.date-badge{{font-family:'IBM Plex Mono',monospace;font-size:.68rem;color:var(--muted);margin-left:auto;text-align:right;line-height:1.8;}}
.updated{{font-size:.6rem;opacity:.6;}}
.tabs{{padding:0 40px;border-bottom:1px solid var(--border);display:flex;overflow-x:auto;gap:0;}}
.tab{{font-family:'IBM Plex Mono',monospace;font-size:.68rem;padding:11px 18px;cursor:pointer;
  color:var(--muted);border-bottom:2px solid transparent;white-space:nowrap;transition:all .15s;}}
.tab:hover{{color:var(--text);}}
.tab.active{{color:var(--accent);border-bottom-color:var(--accent);}}
.main{{padding:28px 40px;display:flex;flex-direction:column;gap:28px;}}
.section-label{{font-family:'IBM Plex Mono',monospace;font-size:.62rem;color:var(--accent);
  letter-spacing:.2em;text-transform:uppercase;margin-bottom:12px;}}
.section-card{{background:var(--panel);border:1px solid var(--border);border-radius:6px;overflow:hidden;margin-bottom:20px;}}
.section-header{{padding:14px 18px;display:flex;align-items:center;gap:10px;border-bottom:1px solid var(--border);}}
.section-dot{{width:8px;height:8px;border-radius:50%;flex-shrink:0;}}
.section-title{{font-family:'Bebas Neue',sans-serif;font-size:1.1rem;letter-spacing:.07em;color:#fff;}}
.data-table{{width:100%;border-collapse:collapse;font-family:'IBM Plex Mono',monospace;font-size:.72rem;}}
.data-table th{{color:var(--muted);text-align:right;padding:7px 14px;border-bottom:1px solid var(--border);
  font-weight:400;letter-spacing:.06em;font-size:.62rem;white-space:nowrap;}}
.data-table th:first-child{{text-align:left;padding-left:18px;}}
.data-table td{{padding:7px 14px;border-bottom:1px solid rgba(30,45,69,.4);text-align:right;color:var(--text);}}
.data-table td:first-child{{text-align:left;padding-left:18px;font-weight:600;}}
.data-table tr.total-row{{cursor:default;}}
.data-table tr.total-row td{{color:#fff;font-weight:600;border-top:1px solid var(--border);border-bottom:1px solid var(--border);}}
.data-table tr.product-row{{cursor:pointer;}}
.data-table tr.product-row:hover{{background:rgba(255,255,255,.02);}}
.data-table tr.product-row td:first-child::before{{content:'▶ ';font-size:.6rem;color:var(--muted);margin-right:4px;}}
.data-table tr.product-row.open td:first-child::before{{content:'▼ ';color:var(--accent);}}
.chart-expand{{display:none;}}
.chart-expand.open{{display:table-row;}}
.chart-expand td{{padding:16px 18px;background:rgba(0,0,0,.2);}}
.chart-wrap{{height:140px;position:relative;}}
.val-pos{{color:#2ecc71;}}
.val-neg{{color:#e74c3c;}}
.val-zero{{color:var(--muted);}}
</style>
</head>
<body>
<header>
  <div class="logo-mark"></div>
  <div>
    <div class="subtitle">S&P Global PointLogic</div>
    <h1>Supply & Demand Dashboard</h1>
  </div>
  <div class="date-badge">
    <div id="dateRange">Loading...</div>
    <div class="updated">Updated {updated}</div>
  </div>
</header>
<div class="tabs" id="tabBar"></div>
<div class="main" id="mainContent"></div>

<script>
{JS_DATA}

let currentRegion = 'Northeast';
let charts = {{}};

function killChart(id) {{ if(charts[id]){{charts[id].destroy();delete charts[id];}} }}
function killAll() {{ Object.keys(charts).forEach(killChart); }}

function fmtDate(d) {{
  return new Date(d+'T00:00:00').toLocaleDateString('en-US',{{month:'short',day:'numeric'}});
}}

function fmtVal(v) {{
  if(v==null||isNaN(v)) return '—';
  return Number(v).toFixed(0);
}}

function valClass(v) {{
  if(v==null||isNaN(v)||v===0) return 'val-zero';
  return v>0?'val-pos':'val-neg';
}}

function categorizeProduct(name) {{
  if(!name) return 'balance';
  const lower = name.toLowerCase();
  
  if(lower.includes('production')) return 'supply';
  if(lower.includes('net inflow')) return 'supply';
  if(lower.includes('flows from')) return 'supply';
  if(lower.includes('withdrawal')) return 'supply';
  
  if(lower.includes('res/com')) return 'demand';
  if(lower.includes('industrial')) return 'demand';
  if(lower.includes('power')) return 'demand';
  if(lower.includes('fuel and pipe')) return 'demand';
  if(lower.includes('net outflow')) return 'demand';
  if(lower.includes('flows to')) return 'demand';
  
  if(lower.includes('total')) return 'balance';
  if(lower.includes('implied')) return 'balance';
  if(lower.includes('balancing')) return 'balance';
  
  return 'other';
}}

function statsForProduct(productName, regionData) {{
  const n = DATES.length;
  const values = DATES.map(d => regionData[d]?.[productName] || null);
  
  const v0 = values[n-1] || 0;
  const v1 = values[n-2] || 0;
  const v2 = values[n-3] || 0;
  
  // WTD
  const latestDate = new Date(DATES[n-1]+'T00:00:00');
  const dow = latestDate.getDay();
  const daysSinceMon = (dow===0)?6:(dow-1);
  const wtdVals = values.slice(Math.max(0,n-1-daysSinceMon)).filter(x=>x!=null);
  const wtd = wtdVals.length ? wtdVals.reduce((a,b)=>a+b,0)/wtdVals.length : 0;
  
  // MTD
  const curMon = DATES[n-1].slice(0,7);
  const mtdVals = DATES.map((d,i)=>d.slice(0,7)===curMon?values[i]:null).filter(x=>x!=null);
  const mtd = mtdVals.length ? mtdVals.reduce((a,b)=>a+b,0)/mtdVals.length : 0;
  
  // YTD
  const curYr = DATES[n-1].slice(0,4);
  const ytdVals = DATES.map((d,i)=>d.slice(0,4)===curYr?values[i]:null).filter(x=>x!=null);
  const ytd = ytdVals.length ? ytdVals.reduce((a,b)=>a+b,0)/ytdVals.length : 0;
  
  return {{v0,v1,v2,wtd,mtd,ytd,values}};
}}

function renderRegion(regionName) {{
  killAll();
  
  const regionId = REGION_IDS[REGION_NAMES.indexOf(regionName)];
  const regionData = DATA[String(regionId)] || {{}};
  const color = REGION_COLORS[regionName];
  
  const n = DATES.length;
  const d0 = DATES[n-1], d1 = DATES[n-2]||d0, d2 = DATES[n-3]||d1;
  
  document.getElementById('dateRange').textContent = 
    DATES[0] + '  →  ' + DATES[DATES.length-1];
  
  // Get all products from latest date
  const latestData = regionData[d0] || {{}};
  
  // Categorize
  const supply = {{}}, demand = {{}}, balance = {{}};
  Object.keys(latestData).forEach(product => {{
    const cat = categorizeProduct(product);
    if(cat === 'supply') supply[product] = true;
    else if(cat === 'demand') demand[product] = true;
    else if(cat === 'balance') balance[product] = true;
  }});
  
  let html = `
    <div class="section-label">// ${{regionName}} — MMcf/d</div>
  `;
  
  // Supply section
  if(Object.keys(supply).length > 0) {{
    html += buildSection('Supply', supply, regionData, '#2ecc71', d0, d1, d2);
  }}
  
  // Demand section
  if(Object.keys(demand).length > 0) {{
    html += buildSection('Demand', demand, regionData, '#ff6b35', d0, d1, d2);
  }}
  
  // Balance section
  if(Object.keys(balance).length > 0) {{
    html += buildSection('Balance & Totals', balance, regionData, '#ffd166', d0, d1, d2);
  }}
  
  document.getElementById('mainContent').innerHTML = html;
}}

function buildSection(sectionName, products, regionData, color, d0, d1, d2) {{
  const productNames = Object.keys(products).sort();
  
  // Calculate totals
  const totalStats = {{v0:0,v1:0,v2:0,wtd:0,mtd:0,ytd:0}};
  productNames.forEach(p => {{
    const stats = statsForProduct(p, regionData);
    totalStats.v0 += stats.v0;
    totalStats.v1 += stats.v1;
    totalStats.v2 += stats.v2;
    totalStats.wtd += stats.wtd;
    totalStats.mtd += stats.mtd;
    totalStats.ytd += stats.ytd;
  }});
  
  let rows = `<tr class="total-row">
    <td>${{sectionName}} Total</td>
    <td class="${{valClass(totalStats.v0)}}">${{fmtVal(totalStats.v0)}}</td>
    <td class="${{valClass(totalStats.v1)}}">${{fmtVal(totalStats.v1)}}</td>
    <td class="${{valClass(totalStats.v2)}}">${{fmtVal(totalStats.v2)}}</td>
    <td class="${{valClass(totalStats.wtd)}}">${{fmtVal(totalStats.wtd)}}</td>
    <td class="${{valClass(totalStats.mtd)}}">${{fmtVal(totalStats.mtd)}}</td>
    <td class="${{valClass(totalStats.ytd)}}">${{fmtVal(totalStats.ytd)}}</td>
  </tr>`;
  
  productNames.forEach(product => {{
    const stats = statsForProduct(product, regionData);
    const rowId = `row_${{sectionName}}_${{product}}`.replace(/[^a-z0-9_]/gi,'_');
    const chartId = `chart_${{sectionName}}_${{product}}`.replace(/[^a-z0-9_]/gi,'_');
    
    rows += `
    <tr class="product-row" id="${{rowId}}" onclick="toggleProductRow('${{rowId}}','${{chartId}}',{{v:${{JSON.stringify(stats.values)}},c:'${{color}}'}})">
      <td>${{product}}</td>
      <td class="${{valClass(stats.v0)}}">${{fmtVal(stats.v0)}}</td>
      <td class="${{valClass(stats.v1)}}">${{fmtVal(stats.v1)}}</td>
      <td class="${{valClass(stats.v2)}}">${{fmtVal(stats.v2)}}</td>
      <td class="${{valClass(stats.wtd)}}">${{fmtVal(stats.wtd)}}</td>
      <td class="${{valClass(stats.mtd)}}">${{fmtVal(stats.mtd)}}</td>
      <td class="${{valClass(stats.ytd)}}">${{fmtVal(stats.ytd)}}</td>
    </tr>
    <tr class="chart-expand" id="exp_${{chartId}}">
      <td colspan="7">
        <div class="chart-wrap"><canvas id="${{chartId}}"></canvas></div>
      </td>
    </tr>`;
  }});
  
  return `
  <div class="section-card" style="border-top:2px solid ${{color}}">
    <div class="section-header">
      <div class="section-dot" style="background:${{color}}"></div>
      <div class="section-title">${{sectionName}}</div>
    </div>
    <div style="overflow-x:auto;">
      <table class="data-table">
        <thead><tr>
          <th>Product</th>
          <th>${{fmtDate(d0)}}</th>
          <th>${{fmtDate(d1)}}</th>
          <th>${{fmtDate(d2)}}</th>
          <th>WTD Avg</th>
          <th>MTD Avg</th>
          <th>YTD Avg</th>
        </tr></thead>
        <tbody>${{rows}}</tbody>
      </table>
    </div>
  </div>`;
}}

function toggleProductRow(rowId, chartId, data) {{
  const row = document.getElementById(rowId);
  const expRow = document.getElementById('exp_'+chartId);
  const isOpen = row.classList.contains('open');
  
  if(isOpen) {{
    row.classList.remove('open');
    expRow.classList.remove('open');
    killChart(chartId);
  }} else {{
    row.classList.add('open');
    expRow.classList.add('open');
    buildLineChart(chartId, data.v, data.c);
  }}
}}

function buildLineChart(canvasId, values, color) {{
  killChart(canvasId);
  const ctx = document.getElementById(canvasId);
  if(!ctx) return;
  
  // Last 14 days only
  const last14Dates = DATES.slice(-14);
  const last14Values = values.slice(-14);
  const labels = last14Dates.map(fmtDate);
  
  const c2 = ctx.getContext('2d');
  const grad = c2.createLinearGradient(0,0,0,140);
  const rgb = color.replace('#','');
  const r = parseInt(rgb.substr(0,2),16);
  const g = parseInt(rgb.substr(2,2),16);
  const b = parseInt(rgb.substr(4,2),16);
  grad.addColorStop(0, `rgba(${{r}},${{g}},${{b}},.22)`);
  grad.addColorStop(1, `rgba(${{r}},${{g}},${{b}},0)`);
  
  charts[canvasId] = new Chart(c2, {{
    type: 'line',
    data: {{
      labels,
      datasets: [{{
        data: last14Values,
        borderColor: color,
        backgroundColor: grad,
        borderWidth: 1.5,
        fill: true,
        tension: 0.3,
        pointRadius: 0,
        pointHoverRadius: 3,
        spanGaps: true
      }}]
    }},
    options: {{
      responsive: true,
      maintainAspectRatio: false,
      interaction: {{mode:'index',intersect:false}},
      plugins: {{
        legend: {{display:false}},
        tooltip: {{
          backgroundColor:'#0d1420',
          borderColor:'#1e2d45',
          borderWidth:1,
          titleFont:{{family:'IBM Plex Mono',size:10}},
          bodyFont:{{family:'IBM Plex Mono',size:10}},
          callbacks:{{label:c=>`${{c.parsed.y?.toFixed(0)||0}} MMcf/d`}}
        }}
      }},
      scales: {{
        x: {{grid:{{color:'rgba(255,255,255,0.04)'}},ticks:{{color:'#64748b',font:{{family:'IBM Plex Mono',size:9}},maxTicksLimit:8}},border:{{display:false}}}},
        y: {{grid:{{color:'rgba(255,255,255,0.04)'}},ticks:{{color:'#64748b',font:{{family:'IBM Plex Mono',size:9}}}},border:{{display:false}}}}
      }}
    }}
  }});
}}

function buildTabs() {{
  const bar = document.getElementById('tabBar');
  bar.innerHTML = '';
  REGION_NAMES.forEach(rname => {{
    const tab = document.createElement('div');
    tab.className = 'tab' + (rname===currentRegion?' active':'');
    tab.textContent = rname;
    if(rname===currentRegion) {{
      tab.style.borderBottomColor = REGION_COLORS[rname];
      tab.style.color = REGION_COLORS[rname];
    }}
    tab.onclick = () => {{ currentRegion=rname; buildTabs(); renderRegion(rname); }};
    bar.appendChild(tab);
  }});
}}

buildTabs();
renderRegion(currentRegion);
</script>
</body>
</html>"""
    
    with open(HTML_FILE, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"   ✓ Dashboard saved to {HTML_FILE}")
    
    # Also copy to SupplyDemand repo as index.html
    print(f"   Copying dashboard to: {DEPLOY_HTML}")
    with open(DEPLOY_HTML, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"   ✓ Dashboard copied to {DEPLOY_HTML}")
    
    # Verify files exist
    if os.path.exists(DEPLOY_HTML):
        size = os.path.getsize(DEPLOY_HTML) / 1024
        print(f"   ✓ Verified: index.html ({size:.1f} KB)")
    if os.path.exists(DEPLOY_DATA):
        size = os.path.getsize(DEPLOY_DATA) / 1024
        print(f"   ✓ Verified: supply_demand_data.json ({size:.1f} KB)")


def check_git_installed():
    """Check if Git is installed."""
    try:
        subprocess.run(["git", "--version"], capture_output=True, text=True, check=True)
        return True
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False


def git_commit_and_push():
    """Commit and push to SupplyDemand GitHub repo."""
    print("\n🔄 Pushing to GitHub (SupplyDemand repo)...")
    
    if not check_git_installed():
        print("\n   ⚠️  Git not installed")
        return False
    
    # Change to SupplyDemand repo
    os.chdir(SUPPLY_DEMAND_REPO)
    
    # Check if git is initialized
    git_check = subprocess.run(["git", "rev-parse", "--git-dir"], capture_output=True, text=True)
    if git_check.returncode != 0:
        print("\n   ⚠️  SupplyDemand folder is not a Git repository")
        print("   Run these commands to set up:")
        print(f"   cd {SUPPLY_DEMAND_REPO}")
        print("   git init")
        print("   git remote add origin https://github.com/ava-ms/SupplyDemand.git")
        print("   git add .")
        print('   git commit -m "Initial commit"')
        print("   git branch -M main")
        print("   git push -u origin main")
        return False
    
    subprocess.run(["git", "config", "user.name", "Supply Demand Bot"], check=False)
    subprocess.run(["git", "config", "user.email", "ava-ms@github.com"], check=False)
    subprocess.run(["git", "add", "."], capture_output=True, text=True)
    
    status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if not status.stdout.strip():
        print("   ℹ️  No changes to commit")
        return True
    
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    commit_msg = f"Auto-update supply/demand dashboard - {timestamp}"
    
    try:
        subprocess.run(["git", "commit", "-m", commit_msg], check=True, capture_output=True, text=True)
        subprocess.run(["git", "push"], capture_output=True, text=True, check=True)
        print(f"   ✓ Successfully pushed to SupplyDemand repo")
        return True
    except:
        print(f"   ⚠️  Push failed")
        return False


def main():
    print("=" * 60)
    print("  Supply & Demand Dashboard — Auto-Update & Deploy")
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    try:
        print("\n[1/4] Fetching supply/demand data...")
        data, dates = fetch_supply_demand_data()

        print("\n[2/4] Saving data...")
        save_data(data)

        print("\n[3/4] Generating dashboard...")
        generate_html(data, dates)

        print("\n[4/4] Deploying to GitHub...")
        git_success = git_commit_and_push()

        print("\n" + "=" * 60)
        if git_success:
            print("  ✅ Supply/demand dashboard updated and deployed!")
            print(f"  📂 Working copy: {HTML_FILE}")
            print(f"  📂 Deployed to: {DEPLOY_HTML}")
            print(f"  🌐 GitHub: https://github.com/ava-ms/SupplyDemand")
            print(f"  🌐 Live: https://ava-ms.github.io/SupplyDemand/")
        else:
            print("  ⚠️  Dashboard updated locally")
            print(f"  📂 Working copy: {HTML_FILE}")
            print(f"  📂 Deployed to: {DEPLOY_HTML}")
        print("=" * 60)

    except Exception as e:
        print(f"\n❌ Error: {e}")
        raise


if __name__ == "__main__":
    main()


  Supply & Demand Dashboard — Auto-Update & Deploy
  2026-03-17 14:44:10

[1/4] Fetching supply/demand data...
📡 Fetching supply/demand data for 8 regions (30 days)...

   Canada (id=26158)
      Done - 30 days fetched              

   Mexico (id=26160)


KeyboardInterrupt: 

In [2]:
"""
Supply/Demand Dashboard - Auto-Update & Deploy
===============================================
Fetches supply/demand data for all regions and generates an interactive dashboard
"""

import requests
import json
import os
import subprocess
from datetime import datetime, timedelta
from collections import defaultdict

# ── CONFIG ────────────────────────────────────────────────────────────────────
USERNAME = "39a9cfa5-18cf-4f0b-b66c-1634b79e906e"
PASSWORD = "cpLScP8pMDGywJks"
AUTH = (USERNAME, PASSWORD)
BASE_URL = "https://api.connect.spglobal.com/cs/v1/pointlogic"

REPO_DIR = r"C:\Users\melaea\LNGFeedgas"
SUPPLY_DEMAND_REPO = r"C:\Users\melaea\LNGFeedGas"
DATA_FILE = os.path.join(REPO_DIR, "supply_demand_data.json")
HTML_FILE = os.path.join(REPO_DIR, "supply_demand_dashboard.html")
EXCEL_FILE = os.path.join(SUPPLY_DEMAND_REPO, "supply_demand_data.xlsx")
DEPLOY_HTML = os.path.join(SUPPLY_DEMAND_REPO, "index.html")
DEPLOY_DATA = os.path.join(SUPPLY_DEMAND_REPO, "supply_demand_data.json")

NUM_DAYS = 30
# ─────────────────────────────────────────────────────────────────────────────

REGIONS = [
    {"id": 26158, "name": "Canada"},
    {"id": 26160, "name": "Mexico"},
    {"id": 26105, "name": "Mid-Continent"},
    {"id": 26104, "name": "Northeast"},
    {"id": 26106, "name": "Rocky Mountain"},
    {"id": 26107, "name": "Southeast"},
    {"id": 26108, "name": "Texas"},
    {"id": 26109, "name": "Western"},
]

REGION_COLORS = {
    "Canada": "#a78bfa",
    "Mexico": "#fb8500",
    "Mid-Continent": "#2ecc71",
    "Northeast": "#00b4d8",
    "Rocky Mountain": "#ffd166",
    "Southeast": "#ff6b35",
    "Texas": "#f72585",
    "Western": "#4cc9f0",
}

REGION_EXCEL_COLORS = {k: v.lstrip("#") for k, v in REGION_COLORS.items()}


def fmt_date(dt):
    return f"{dt.month}/{dt.day}/{dt.year}"


def fetch_supply_demand_data():
    print(f"📡 Fetching supply/demand data for {len(REGIONS)} regions ({NUM_DAYS} days)...")
    today = datetime.today()
    dates = [(today - timedelta(days=i)).strftime("%Y-%m-%d")
             for i in range(NUM_DAYS - 1, -1, -1)]
    all_data = {}

    for region in REGIONS:
        rid = region["id"]
        rname = region["name"]
        print(f"\n   {rname} (id={rid})")
        all_data[str(rid)] = {}

        for date_str in dates:
            dt = datetime.strptime(date_str, "%Y-%m-%d")
            report_date = fmt_date(dt)
            url = f"{BASE_URL}/supplyDemand/region/{rid}"
            params = {"reportDate": report_date}
            try:
                resp = requests.get(url, auth=AUTH, params=params, timeout=30)
                resp.raise_for_status()
                data = resp.json().get("Data", [])
                products = {}
                for record in data:
                    product = record.get("product")
                    volume = record.get("volume_mmcfd", 0)
                    if product:
                        products[product] = volume
                all_data[str(rid)][date_str] = products
                print(f"      {date_str}: {len(products)} products", end="\r")
            except Exception as e:
                print(f"      {date_str}: ERROR - {e}")
                all_data[str(rid)][date_str] = {}

        print(f"      Done - {len(dates)} days fetched              ")

    print(f"\n   ✓ All regions fetched")
    return all_data, dates


def save_data(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=2)
    print(f"   ✓ Data saved to {DATA_FILE}")
    os.makedirs(SUPPLY_DEMAND_REPO, exist_ok=True)
    with open(DEPLOY_DATA, "w") as f:
        json.dump(data, f, indent=2)
    print(f"   ✓ Data copied to {DEPLOY_DATA}")


# ── Excel Export ──────────────────────────────────────────────────────────────
def categorize_product(name):
    if not name:
        return "balance"
    lower = name.lower()
    if any(x in lower for x in ["production", "net inflow", "flows from", "withdrawal"]):
        return "supply"
    if any(x in lower for x in ["res/com", "industrial", "power", "fuel and pipe", "net outflow", "flows to"]):
        return "demand"
    if any(x in lower for x in ["total", "implied", "balancing"]):
        return "balance"
    return "other"


def export_to_excel(data, dates):
    print("   Generating Excel export...")
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter
    except ImportError:
        print("   ⚠️  openpyxl not installed. Run: pip install openpyxl")
        return

    # Style helpers
    BORDER_BOTTOM = Border(bottom=Side(style="thin", color="1e2d45"))
    NUM_FMT = "#,##0.0"

    def hdr_cell(cell, bg="0a0e17"):
        cell.font = Font(name="Arial", bold=True, color="E2E8F0", size=9)
        cell.fill = PatternFill("solid", fgColor=bg)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    def data_cell(cell, alt=False, bold=False, color="C8D8E8"):
        cell.font = Font(name="Arial", bold=bold, size=9, color="FFFFFF" if bold else color)
        cell.fill = PatternFill("solid", fgColor="1e2d45" if bold else ("0d1420" if alt else "111827"))
        cell.alignment = Alignment(horizontal="right", vertical="center")
        cell.border = BORDER_BOTTOM

    def label_cell(cell, bold=False, indent=0):
        pad = "  " * indent
        cell.value = pad + (cell.value or "")
        cell.font = Font(name="Arial", bold=bold, size=9, color="FFFFFF" if bold else "C8D8E8")
        cell.fill = PatternFill("solid", fgColor="1e2d45" if bold else "111827")
        cell.alignment = Alignment(horizontal="left", vertical="center")
        cell.border = BORDER_BOTTOM

    def avg_vals(vals):
        v = [x for x in vals if x is not None]
        return sum(v) / len(v) if v else 0

    wb = Workbook()

    # ── Summary sheet ──────────────────────────────────────────────────────
    ws_sum = wb.active
    ws_sum.title = "Summary"
    ws_sum.sheet_view.showGridLines = False
    ws_sum.sheet_properties.tabColor = "ff6b35"

    # Title
    ws_sum.merge_cells(f"A1:{get_column_letter(len(dates) + 4)}1")
    tc = ws_sum["A1"]
    tc.value = f"Supply & Demand Summary — All Regions  |  {dates[0]} → {dates[-1]}"
    tc.font = Font(name="Arial", bold=True, size=12, color="FFFFFF")
    tc.fill = PatternFill("solid", fgColor="ff6b35")
    tc.alignment = Alignment(horizontal="left", vertical="center")
    ws_sum.row_dimensions[1].height = 22

    # Headers: Region | Product | Category | date1 | date2 | ... | WTD | MTD | YTD
    stat_cols = ["WTD Avg", "MTD Avg", "YTD Avg"]
    headers = ["Region", "Product", "Category"] + [
        datetime.strptime(d, "%Y-%m-%d").strftime("%b %-d") if os.name != "nt"
        else datetime.strptime(d, "%Y-%m-%d").strftime("%b %#d")
        for d in dates
    ] + stat_cols

    for ci, h in enumerate(headers, start=1):
        cell = ws_sum.cell(row=2, column=ci, value=h)
        hdr_cell(cell)
        ws_sum.column_dimensions[get_column_letter(ci)].width = 12 if ci > 3 else (20 if ci <= 2 else 10)
    ws_sum.row_dimensions[2].height = 26

    ri = 3
    latest_mon = dates[-1][:7]
    latest_yr = dates[-1][:4]

    for region in REGIONS:
        rid = str(region["id"])
        rname = region["name"]
        color = REGION_EXCEL_COLORS[rname]
        region_data = data.get(rid, {})

        # Collect all products across all dates
        all_products = set()
        for d in dates:
            all_products.update(region_data.get(d, {}).keys())

        for product in sorted(all_products):
            cat = categorize_product(product)
            vals = [region_data.get(d, {}).get(product) for d in dates]
            wtd_days = min(7, len(dates))
            wtd = avg_vals(vals[-wtd_days:])
            mtd = avg_vals([vals[i] for i, d in enumerate(dates) if d[:7] == latest_mon])
            ytd = avg_vals([vals[i] for i, d in enumerate(dates) if d[:4] == latest_yr])

            alt = (ri % 2 == 0)
            ws_sum.cell(row=ri, column=1, value=rname).font = Font(name="Arial", size=9,
                color=color, bold=True)
            ws_sum.cell(row=ri, column=1).fill = PatternFill("solid", fgColor="0d1420" if alt else "111827")
            ws_sum.cell(row=ri, column=1).alignment = Alignment(horizontal="left", vertical="center")
            ws_sum.cell(row=ri, column=1).border = BORDER_BOTTOM

            ws_sum.cell(row=ri, column=2, value=product)
            label_cell(ws_sum.cell(row=ri, column=2))
            ws_sum.cell(row=ri, column=3, value=cat.title())
            label_cell(ws_sum.cell(row=ri, column=3))

            for ci, v in enumerate(vals, start=4):
                cell = ws_sum.cell(row=ri, column=ci, value=v)
                data_cell(cell, alt=alt)
                if v is not None:
                    cell.number_format = NUM_FMT

            for ci, v in enumerate([wtd, mtd, ytd], start=4 + len(dates)):
                cell = ws_sum.cell(row=ri, column=ci, value=v)
                data_cell(cell, alt=alt)
                cell.number_format = NUM_FMT

            ws_sum.row_dimensions[ri].height = 14
            ri += 1

    ws_sum.freeze_panes = "D3"
    ws_sum.auto_filter.ref = f"A2:{get_column_letter(len(headers))}2"

    # ── One sheet per region ───────────────────────────────────────────────
    for region in REGIONS:
        rid = str(region["id"])
        rname = region["name"]
        color = REGION_EXCEL_COLORS[rname]
        region_data = data.get(rid, {})

        ws = wb.create_sheet(title=rname[:31])
        ws.sheet_view.showGridLines = False
        ws.sheet_properties.tabColor = color

        # Title
        ncols = len(dates) + 4
        ws.merge_cells(f"A1:{get_column_letter(ncols)}1")
        tc = ws["A1"]
        tc.value = f"{rname} — Supply & Demand  |  {dates[0]} → {dates[-1]}  |  MMcf/d"
        tc.font = Font(name="Arial", bold=True, size=12, color="FFFFFF")
        tc.fill = PatternFill("solid", fgColor=color)
        tc.alignment = Alignment(horizontal="left", vertical="center")
        ws.row_dimensions[1].height = 22

        date_labels = [
            datetime.strptime(d, "%Y-%m-%d").strftime("%b %-d") if os.name != "nt"
            else datetime.strptime(d, "%Y-%m-%d").strftime("%b %#d")
            for d in dates
        ]
        headers = ["Category", "Product"] + date_labels + stat_cols
        for ci, h in enumerate(headers, start=1):
            cell = ws.cell(row=2, column=ci, value=h)
            hdr_cell(cell, bg=color)
            ws.column_dimensions[get_column_letter(ci)].width = 12 if ci > 2 else (10 if ci == 1 else 28)
        ws.row_dimensions[2].height = 26

        # Collect and group products
        all_products = set()
        for d in dates:
            all_products.update(region_data.get(d, {}).keys())

        grouped = {"supply": [], "demand": [], "balance": [], "other": []}
        for p in sorted(all_products):
            grouped[categorize_product(p)].append(p)

        ri = 3
        section_labels = {"supply": "Supply", "demand": "Demand",
                          "balance": "Balance & Totals", "other": "Other"}
        section_colors = {"supply": "2ecc71", "demand": "ff6b35",
                          "balance": "ffd166", "other": "64748b"}

        for cat_key, products in grouped.items():
            if not products:
                continue

            sec_color = section_colors[cat_key]
            sec_label = section_labels[cat_key]

            # Section header row
            ws.merge_cells(f"A{ri}:{get_column_letter(ncols)}{ri}")
            sc = ws.cell(row=ri, column=1, value=f"── {sec_label} ──")
            sc.font = Font(name="Arial", bold=True, size=9, color="FFFFFF")
            sc.fill = PatternFill("solid", fgColor=sec_color)
            sc.alignment = Alignment(horizontal="left", vertical="center")
            ws.row_dimensions[ri].height = 16
            ri += 1

            # Section total row
            total_vals = []
            for di in range(len(dates)):
                total_vals.append(sum(
                    region_data.get(dates[di], {}).get(p, 0) or 0
                    for p in products
                ))
            wtd = avg_vals(total_vals[-7:])
            mtd = avg_vals([total_vals[i] for i, d in enumerate(dates) if d[:7] == latest_mon])
            ytd = avg_vals([total_vals[i] for i, d in enumerate(dates) if d[:4] == latest_yr])

            ws.cell(row=ri, column=1, value=sec_label + " Total")
            ws.cell(row=ri, column=2, value="")
            for col in range(1, 3):
                c = ws.cell(row=ri, column=col)
                c.font = Font(name="Arial", bold=True, size=9, color="FFFFFF")
                c.fill = PatternFill("solid", fgColor="1e2d45")
                c.alignment = Alignment(horizontal="left" if col < 3 else "right", vertical="center")
                c.border = BORDER_BOTTOM
            for ci, v in enumerate(total_vals, start=3):
                cell = ws.cell(row=ri, column=ci, value=v)
                data_cell(cell, bold=True)
                cell.number_format = NUM_FMT
            for ci, v in enumerate([wtd, mtd, ytd], start=3 + len(dates)):
                cell = ws.cell(row=ri, column=ci, value=v)
                data_cell(cell, bold=True)
                cell.number_format = NUM_FMT
            ws.row_dimensions[ri].height = 16
            ri += 1

            # Individual product rows
            for pi, product in enumerate(products):
                vals = [region_data.get(d, {}).get(product) for d in dates]
                wtd = avg_vals(vals[-7:])
                mtd = avg_vals([vals[i] for i, d in enumerate(dates) if d[:7] == latest_mon])
                ytd = avg_vals([vals[i] for i, d in enumerate(dates) if d[:4] == latest_yr])
                alt = (pi % 2 == 0)

                ws.cell(row=ri, column=1, value=sec_label)
                c1 = ws.cell(row=ri, column=1)
                c1.font = Font(name="Arial", size=8, color="64748b")
                c1.fill = PatternFill("solid", fgColor="0d1420" if alt else "111827")
                c1.alignment = Alignment(horizontal="left", vertical="center")
                c1.border = BORDER_BOTTOM

                ws.cell(row=ri, column=2, value=product)
                label_cell(ws.cell(row=ri, column=2))

                for ci, v in enumerate(vals, start=3):
                    cell = ws.cell(row=ri, column=ci, value=v)
                    data_cell(cell, alt=alt)
                    if v is not None:
                        cell.number_format = NUM_FMT
                for ci, v in enumerate([wtd, mtd, ytd], start=3 + len(dates)):
                    cell = ws.cell(row=ri, column=ci, value=v)
                    data_cell(cell, alt=alt)
                    cell.number_format = NUM_FMT

                ws.row_dimensions[ri].height = 14
                ri += 1

        ws.freeze_panes = f"C3"

    # ── Raw Data sheet ─────────────────────────────────────────────────────
    ws_raw = wb.create_sheet(title="Raw Data")
    ws_raw.sheet_view.showGridLines = False
    ws_raw.sheet_properties.tabColor = "64748b"

    raw_headers = ["region", "date", "product", "category", "volume_mmcfd"]
    for ci, h in enumerate(raw_headers, start=1):
        cell = ws_raw.cell(row=1, column=ci, value=h)
        hdr_cell(cell, bg="1e2d45")
        ws_raw.column_dimensions[get_column_letter(ci)].width = 22

    ws_raw.row_dimensions[1].height = 22
    ri = 2
    for region in REGIONS:
        rid = str(region["id"])
        rname = region["name"]
        region_data = data.get(rid, {})
        for d in dates:
            for product, volume in sorted(region_data.get(d, {}).items()):
                alt = (ri % 2 == 0)
                row_vals = [rname, d, product, categorize_product(product).title(), volume]
                for ci, v in enumerate(row_vals, start=1):
                    cell = ws_raw.cell(row=ri, column=ci, value=v)
                    cell.font = Font(name="Arial", size=9, color="C8D8E8")
                    cell.fill = PatternFill("solid", fgColor="0d1420" if alt else "111827")
                    cell.alignment = Alignment(
                        horizontal="right" if ci == 5 else "left", vertical="center")
                    cell.border = BORDER_BOTTOM
                    if ci == 5 and v is not None:
                        cell.number_format = NUM_FMT
                ws_raw.row_dimensions[ri].height = 14
                ri += 1

    ws_raw.freeze_panes = "A2"
    ws_raw.auto_filter.ref = f"A1:{get_column_letter(len(raw_headers))}1"

    wb.save(EXCEL_FILE)
    print(f"   ✓ Excel export saved to {EXCEL_FILE}")


# ── HTML Generation ───────────────────────────────────────────────────────────
def generate_html(data, dates):
    print("   Generating dashboard HTML...")
    updated = datetime.now().strftime("%B %#d, %Y at %I:%M %p")

    js_data_lines = []
    js_data_lines.append(f"const DATES = {json.dumps(dates)};")
    js_data_lines.append(f"const REGION_NAMES = {json.dumps([r['name'] for r in REGIONS])};")
    js_data_lines.append(f"const REGION_IDS = {json.dumps([r['id'] for r in REGIONS])};")
    js_data_lines.append(f"const REGION_COLORS = {json.dumps(REGION_COLORS)};")
    js_data_lines.append(f"const DATA = {json.dumps(data)};")
    JS_DATA = "\n".join(js_data_lines)

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Supply & Demand Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Bebas+Neue&family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;600&display=swap" rel="stylesheet">
<style>
:root {{
  --bg:#0a0e17; --panel:#111827; --border:#1e2d45;
  --accent:#00b4d8; --text:#e2e8f0; --muted:#64748b;
  --supply:#2ecc71; --demand:#ff6b35; --balance:#ffd166;
}}
*{{margin:0;padding:0;box-sizing:border-box;}}
body{{background:var(--bg);color:var(--text);font-family:'IBM Plex Sans',sans-serif;min-height:100vh;
  background-image:radial-gradient(ellipse at 20% 0%,rgba(0,212,255,.06) 0%,transparent 50%),
  radial-gradient(ellipse at 80% 100%,rgba(255,107,53,.05) 0%,transparent 50%);}}
header{{padding:24px 40px 16px;border-bottom:1px solid var(--border);display:flex;align-items:flex-end;gap:20px;}}
.logo-mark{{width:4px;height:44px;background:linear-gradient(180deg,#2ecc71,#ff6b35);border-radius:2px;flex-shrink:0;}}
h1{{font-family:'Bebas Neue',sans-serif;font-size:2.6rem;letter-spacing:.08em;line-height:1;color:#fff;}}
.subtitle{{font-family:'IBM Plex Mono',monospace;font-size:.68rem;color:var(--accent);letter-spacing:.15em;text-transform:uppercase;margin-bottom:3px;}}
.date-badge{{font-family:'IBM Plex Mono',monospace;font-size:.68rem;color:var(--muted);margin-left:auto;text-align:right;line-height:1.8;}}
.updated{{font-size:.6rem;opacity:.6;}}
.tabs{{padding:0 40px;border-bottom:1px solid var(--border);display:flex;overflow-x:auto;gap:0;}}
.tab{{font-family:'IBM Plex Mono',monospace;font-size:.68rem;padding:11px 18px;cursor:pointer;
  color:var(--muted);border-bottom:2px solid transparent;white-space:nowrap;transition:all .15s;}}
.tab:hover{{color:var(--text);}}
.tab.active{{color:var(--accent);border-bottom-color:var(--accent);}}
.main{{padding:28px 40px;display:flex;flex-direction:column;gap:28px;}}
.section-label{{font-family:'IBM Plex Mono',monospace;font-size:.62rem;color:var(--accent);
  letter-spacing:.2em;text-transform:uppercase;margin-bottom:12px;}}
.section-card{{background:var(--panel);border:1px solid var(--border);border-radius:6px;overflow:hidden;margin-bottom:20px;}}
.section-header{{padding:14px 18px;display:flex;align-items:center;gap:10px;border-bottom:1px solid var(--border);}}
.section-dot{{width:8px;height:8px;border-radius:50%;flex-shrink:0;}}
.section-title{{font-family:'Bebas Neue',sans-serif;font-size:1.1rem;letter-spacing:.07em;color:#fff;}}
.data-table{{width:100%;border-collapse:collapse;font-family:'IBM Plex Mono',monospace;font-size:.72rem;}}
.data-table th{{color:var(--muted);text-align:right;padding:7px 14px;border-bottom:1px solid var(--border);
  font-weight:400;letter-spacing:.06em;font-size:.62rem;white-space:nowrap;}}
.data-table th:first-child{{text-align:left;padding-left:18px;}}
.data-table td{{padding:7px 14px;border-bottom:1px solid rgba(30,45,69,.4);text-align:right;color:var(--text);}}
.data-table td:first-child{{text-align:left;padding-left:18px;font-weight:600;}}
.data-table tr.total-row{{cursor:default;}}
.data-table tr.total-row td{{color:#fff;font-weight:600;border-top:1px solid var(--border);border-bottom:1px solid var(--border);}}
.data-table tr.product-row{{cursor:pointer;}}
.data-table tr.product-row:hover{{background:rgba(255,255,255,.02);}}
.data-table tr.product-row td:first-child::before{{content:'▶ ';font-size:.6rem;color:var(--muted);margin-right:4px;}}
.data-table tr.product-row.open td:first-child::before{{content:'▼ ';color:var(--accent);}}
.chart-expand{{display:none;}}
.chart-expand.open{{display:table-row;}}
.chart-expand td{{padding:16px 18px;background:rgba(0,0,0,.2);}}
.chart-wrap{{height:140px;position:relative;}}
.val-pos{{color:#2ecc71;}}
.val-neg{{color:#e74c3c;}}
.val-zero{{color:var(--muted);}}
</style>
</head>
<body>
<header>
  <div class="logo-mark"></div>
  <div>
    <div class="subtitle">S&P Global PointLogic</div>
    <h1>Supply & Demand Dashboard</h1>
  </div>
  <div class="date-badge">
    <div id="dateRange">Loading...</div>
    <div class="updated">Updated {updated}</div>
  </div>
</header>
<div class="tabs" id="tabBar"></div>
<div class="main" id="mainContent"></div>

<script>
{JS_DATA}

let currentRegion = 'Northeast';
let charts = {{}};

function killChart(id) {{ if(charts[id]){{charts[id].destroy();delete charts[id];}} }}
function killAll() {{ Object.keys(charts).forEach(killChart); }}

function fmtDate(d) {{
  return new Date(d+'T00:00:00').toLocaleDateString('en-US',{{month:'short',day:'numeric'}});
}}

function fmtVal(v) {{
  if(v==null||isNaN(v)) return '—';
  return Number(v).toFixed(0);
}}

function valClass(v) {{
  if(v==null||isNaN(v)||v===0) return 'val-zero';
  return v>0?'val-pos':'val-neg';
}}

function categorizeProduct(name) {{
  if(!name) return 'balance';
  const lower = name.toLowerCase();
  if(lower.includes('production')) return 'supply';
  if(lower.includes('net inflow')) return 'supply';
  if(lower.includes('flows from')) return 'supply';
  if(lower.includes('withdrawal')) return 'supply';
  if(lower.includes('res/com')) return 'demand';
  if(lower.includes('industrial')) return 'demand';
  if(lower.includes('power')) return 'demand';
  if(lower.includes('fuel and pipe')) return 'demand';
  if(lower.includes('net outflow')) return 'demand';
  if(lower.includes('flows to')) return 'demand';
  if(lower.includes('total')) return 'balance';
  if(lower.includes('implied')) return 'balance';
  if(lower.includes('balancing')) return 'balance';
  return 'other';
}}

function statsForProduct(productName, regionData) {{
  const n = DATES.length;
  const values = DATES.map(d => regionData[d]?.[productName] || null);
  const v0 = values[n-1] || 0;
  const v1 = values[n-2] || 0;
  const v2 = values[n-3] || 0;
  const latestDate = new Date(DATES[n-1]+'T00:00:00');
  const dow = latestDate.getDay();
  const daysSinceMon = (dow===0)?6:(dow-1);
  const wtdVals = values.slice(Math.max(0,n-1-daysSinceMon)).filter(x=>x!=null);
  const wtd = wtdVals.length ? wtdVals.reduce((a,b)=>a+b,0)/wtdVals.length : 0;
  const curMon = DATES[n-1].slice(0,7);
  const mtdVals = DATES.map((d,i)=>d.slice(0,7)===curMon?values[i]:null).filter(x=>x!=null);
  const mtd = mtdVals.length ? mtdVals.reduce((a,b)=>a+b,0)/mtdVals.length : 0;
  const curYr = DATES[n-1].slice(0,4);
  const ytdVals = DATES.map((d,i)=>d.slice(0,4)===curYr?values[i]:null).filter(x=>x!=null);
  const ytd = ytdVals.length ? ytdVals.reduce((a,b)=>a+b,0)/ytdVals.length : 0;
  return {{v0,v1,v2,wtd,mtd,ytd,values}};
}}

function renderRegion(regionName) {{
  killAll();
  const regionId = REGION_IDS[REGION_NAMES.indexOf(regionName)];
  const regionData = DATA[String(regionId)] || {{}};
  const color = REGION_COLORS[regionName];
  const n = DATES.length;
  const d0 = DATES[n-1], d1 = DATES[n-2]||d0, d2 = DATES[n-3]||d1;
  document.getElementById('dateRange').textContent = DATES[0] + '  →  ' + DATES[DATES.length-1];
  const latestData = regionData[d0] || {{}};
  const supply = {{}}, demand = {{}}, balance = {{}};
  Object.keys(latestData).forEach(product => {{
    const cat = categorizeProduct(product);
    if(cat === 'supply') supply[product] = true;
    else if(cat === 'demand') demand[product] = true;
    else if(cat === 'balance') balance[product] = true;
  }});
  let html = `<div class="section-label">// ${{regionName}} — MMcf/d</div>`;
  if(Object.keys(supply).length > 0) html += buildSection('Supply', supply, regionData, '#2ecc71', d0, d1, d2);
  if(Object.keys(demand).length > 0) html += buildSection('Demand', demand, regionData, '#ff6b35', d0, d1, d2);
  if(Object.keys(balance).length > 0) html += buildSection('Balance & Totals', balance, regionData, '#ffd166', d0, d1, d2);
  document.getElementById('mainContent').innerHTML = html;
}}

function buildSection(sectionName, products, regionData, color, d0, d1, d2) {{
  const productNames = Object.keys(products).sort();
  const totalStats = {{v0:0,v1:0,v2:0,wtd:0,mtd:0,ytd:0}};
  productNames.forEach(p => {{
    const stats = statsForProduct(p, regionData);
    totalStats.v0 += stats.v0; totalStats.v1 += stats.v1; totalStats.v2 += stats.v2;
    totalStats.wtd += stats.wtd; totalStats.mtd += stats.mtd; totalStats.ytd += stats.ytd;
  }});
  let rows = `<tr class="total-row">
    <td>${{sectionName}} Total</td>
    <td class="${{valClass(totalStats.v0)}}">${{fmtVal(totalStats.v0)}}</td>
    <td class="${{valClass(totalStats.v1)}}">${{fmtVal(totalStats.v1)}}</td>
    <td class="${{valClass(totalStats.v2)}}">${{fmtVal(totalStats.v2)}}</td>
    <td class="${{valClass(totalStats.wtd)}}">${{fmtVal(totalStats.wtd)}}</td>
    <td class="${{valClass(totalStats.mtd)}}">${{fmtVal(totalStats.mtd)}}</td>
    <td class="${{valClass(totalStats.ytd)}}">${{fmtVal(totalStats.ytd)}}</td>
  </tr>`;
  productNames.forEach(product => {{
    const stats = statsForProduct(product, regionData);
    const rowId = `row_${{sectionName}}_${{product}}`.replace(/[^a-z0-9_]/gi,'_');
    const chartId = `chart_${{sectionName}}_${{product}}`.replace(/[^a-z0-9_]/gi,'_');
    rows += `
    <tr class="product-row" id="${{rowId}}" onclick="toggleProductRow('${{rowId}}','${{chartId}}',{{v:${{JSON.stringify(stats.values)}},c:'${{color}}'}})">
      <td>${{product}}</td>
      <td class="${{valClass(stats.v0)}}">${{fmtVal(stats.v0)}}</td>
      <td class="${{valClass(stats.v1)}}">${{fmtVal(stats.v1)}}</td>
      <td class="${{valClass(stats.v2)}}">${{fmtVal(stats.v2)}}</td>
      <td class="${{valClass(stats.wtd)}}">${{fmtVal(stats.wtd)}}</td>
      <td class="${{valClass(stats.mtd)}}">${{fmtVal(stats.mtd)}}</td>
      <td class="${{valClass(stats.ytd)}}">${{fmtVal(stats.ytd)}}</td>
    </tr>
    <tr class="chart-expand" id="exp_${{chartId}}">
      <td colspan="7"><div class="chart-wrap"><canvas id="${{chartId}}"></canvas></div></td>
    </tr>`;
  }});
  return `
  <div class="section-card" style="border-top:2px solid ${{color}}">
    <div class="section-header">
      <div class="section-dot" style="background:${{color}}"></div>
      <div class="section-title">${{sectionName}}</div>
    </div>
    <div style="overflow-x:auto;">
      <table class="data-table">
        <thead><tr>
          <th>Product</th>
          <th>${{fmtDate(d0)}}</th><th>${{fmtDate(d1)}}</th><th>${{fmtDate(d2)}}</th>
          <th>WTD Avg</th><th>MTD Avg</th><th>YTD Avg</th>
        </tr></thead>
        <tbody>${{rows}}</tbody>
      </table>
    </div>
  </div>`;
}}

function toggleProductRow(rowId, chartId, data) {{
  const row = document.getElementById(rowId);
  const expRow = document.getElementById('exp_'+chartId);
  const isOpen = row.classList.contains('open');
  if(isOpen) {{ row.classList.remove('open'); expRow.classList.remove('open'); killChart(chartId); }}
  else {{ row.classList.add('open'); expRow.classList.add('open'); buildLineChart(chartId, data.v, data.c); }}
}}

function buildLineChart(canvasId, values, color) {{
  killChart(canvasId);
  const ctx = document.getElementById(canvasId);
  if(!ctx) return;
  const last14Dates = DATES.slice(-14);
  const last14Values = values.slice(-14);
  const labels = last14Dates.map(fmtDate);
  const c2 = ctx.getContext('2d');
  const grad = c2.createLinearGradient(0,0,0,140);
  const rgb = color.replace('#','');
  const r = parseInt(rgb.substr(0,2),16), g = parseInt(rgb.substr(2,2),16), b = parseInt(rgb.substr(4,2),16);
  grad.addColorStop(0, `rgba(${{r}},${{g}},${{b}},.22)`);
  grad.addColorStop(1, `rgba(${{r}},${{g}},${{b}},0)`);
  charts[canvasId] = new Chart(c2, {{
    type: 'line',
    data: {{ labels, datasets: [{{ data: last14Values, borderColor: color, backgroundColor: grad,
      borderWidth: 1.5, fill: true, tension: 0.3, pointRadius: 0, pointHoverRadius: 3, spanGaps: true }}] }},
    options: {{
      responsive: true, maintainAspectRatio: false,
      interaction: {{mode:'index',intersect:false}},
      plugins: {{ legend:{{display:false}}, tooltip:{{ backgroundColor:'#0d1420', borderColor:'#1e2d45',
        borderWidth:1, titleFont:{{family:'IBM Plex Mono',size:10}}, bodyFont:{{family:'IBM Plex Mono',size:10}},
        callbacks:{{label:c=>`${{c.parsed.y?.toFixed(0)||0}} MMcf/d`}} }} }},
      scales: {{
        x: {{grid:{{color:'rgba(255,255,255,0.04)'}},ticks:{{color:'#64748b',font:{{family:'IBM Plex Mono',size:9}},maxTicksLimit:8}},border:{{display:false}}}},
        y: {{grid:{{color:'rgba(255,255,255,0.04)'}},ticks:{{color:'#64748b',font:{{family:'IBM Plex Mono',size:9}}}},border:{{display:false}}}}
      }}
    }}
  }});
}}

function buildTabs() {{
  const bar = document.getElementById('tabBar');
  bar.innerHTML = '';
  REGION_NAMES.forEach(rname => {{
    const tab = document.createElement('div');
    tab.className = 'tab' + (rname===currentRegion?' active':'');
    tab.textContent = rname;
    if(rname===currentRegion) {{ tab.style.borderBottomColor = REGION_COLORS[rname]; tab.style.color = REGION_COLORS[rname]; }}
    tab.onclick = () => {{ currentRegion=rname; buildTabs(); renderRegion(rname); }};
    bar.appendChild(tab);
  }});
}}

buildTabs();
renderRegion(currentRegion);
</script>
</body>
</html>"""

    with open(HTML_FILE, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"   ✓ Dashboard saved to {HTML_FILE}")
    with open(DEPLOY_HTML, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"   ✓ Dashboard copied to {DEPLOY_HTML}")
    if os.path.exists(DEPLOY_HTML):
        print(f"   ✓ Verified: index.html ({os.path.getsize(DEPLOY_HTML)/1024:.1f} KB)")
    if os.path.exists(DEPLOY_DATA):
        print(f"   ✓ Verified: supply_demand_data.json ({os.path.getsize(DEPLOY_DATA)/1024:.1f} KB)")


# ── Git ───────────────────────────────────────────────────────────────────────
def check_git_installed():
    try:
        subprocess.run(["git", "--version"], capture_output=True, text=True, check=True)
        return True
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False


def git_commit_and_push():
    print("\n🔄 Pushing to GitHub (SupplyDemand repo)...")
    if not check_git_installed():
        print("\n   ⚠️  Git not installed")
        return False

    os.chdir(SUPPLY_DEMAND_REPO)
    git_check = subprocess.run(["git", "rev-parse", "--git-dir"], capture_output=True, text=True)
    if git_check.returncode != 0:
        print("\n   ⚠️  SupplyDemand folder is not a Git repository")
        print("   Run these commands to set up:")
        print(f"   cd {SUPPLY_DEMAND_REPO}")
        print("   git init")
        print("   git remote add origin https://github.com/ava-ms/SupplyDemand.git")
        print("   git add .")
        print('   git commit -m "Initial commit"')
        print("   git branch -M main")
        print("   git push -u origin main")
        return False

    subprocess.run(["git", "config", "user.name", "Supply Demand Bot"], check=False)
    subprocess.run(["git", "config", "user.email", "ava-ms@github.com"], check=False)
    subprocess.run(["git", "add", "."], capture_output=True, text=True)

    status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if not status.stdout.strip():
        print("   ℹ️  No changes to commit")
        return True

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    try:
        subprocess.run(["git", "commit", "-m", f"Auto-update supply/demand dashboard - {timestamp}"],
                       check=True, capture_output=True, text=True)
        subprocess.run(["git", "push"], capture_output=True, text=True, check=True)
        print(f"   ✓ Successfully pushed to SupplyDemand repo")
        return True
    except:
        print(f"   ⚠️  Push failed")
        return False


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  Supply & Demand Dashboard — Auto-Update & Deploy")
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    try:
        print("\n[1/5] Fetching supply/demand data...")
        data, dates = fetch_supply_demand_data()

        print("\n[2/5] Saving data...")
        save_data(data)

        print("\n[3/5] Generating dashboard HTML...")
        generate_html(data, dates)

        print("\n[4/5] Exporting to Excel...")
        export_to_excel(data, dates)

        print("\n[5/5] Deploying to GitHub...")
        git_success = git_commit_and_push()

        print("\n" + "=" * 60)
        if git_success:
            print("  ✅ Supply/demand dashboard updated and deployed!")
            print(f"  📂 Working copy: {HTML_FILE}")
            print(f"  📊 Excel export: {EXCEL_FILE}")
            print(f"  📂 Deployed to:  {DEPLOY_HTML}")
            print(f"  🌐 GitHub:       https://github.com/ava-ms/SupplyDemand")
            print(f"  🌐 Live:         https://ava-ms.github.io/SupplyDemand/")
        else:
            print("  ⚠️  Dashboard updated locally")
            print(f"  📂 Working copy: {HTML_FILE}")
            print(f"  📊 Excel export: {EXCEL_FILE}")
            print(f"  📂 Deployed to:  {DEPLOY_HTML}")
        print("=" * 60)

    except Exception as e:
        print(f"\n❌ Error: {e}")
        raise


if __name__ == "__main__":
    main()


# ══════════════════════════════════════════════════════════════════════════════
# USAGE
# ══════════════════════════════════════════════════════════════════════════════
#
#     python update_supply_demand_dashboard.py
#
# Requirements:
#     pip install requests openpyxl
#
# This will automatically:
#   1. Fetch supply/demand data from S&P Global (all regions, last 30 days)
#   2. Save raw JSON data
#   3. Generate supply_demand_dashboard.html
#   4. Export supply_demand_data.xlsx with:
#        - Summary sheet (all regions × all products, dates as columns)
#        - One sheet per region (Supply / Demand / Balance sections)
#        - Raw Data sheet (flat, filterable)
#   5. Commit and push everything to GitHub → SupplyDemand repo
#
# ══════════════════════════════════════════════════════════════════════════════

  Supply & Demand Dashboard — Auto-Update & Deploy
  2026-03-19 15:21:22

[1/5] Fetching supply/demand data...
📡 Fetching supply/demand data for 8 regions (30 days)...

   Canada (id=26158)
      Done - 30 days fetched              

   Mexico (id=26160)
      Done - 30 days fetched              

   Mid-Continent (id=26105)
      Done - 30 days fetched              

   Northeast (id=26104)
      Done - 30 days fetched              

   Rocky Mountain (id=26106)
      Done - 30 days fetched              

   Southeast (id=26107)
      Done - 30 days fetched              

   Texas (id=26108)
      Done - 30 days fetched              

   Western (id=26109)
      Done - 30 days fetched              

   ✓ All regions fetched

[2/5] Saving data...
   ✓ Data saved to C:\Users\melaea\LNGFeedgas\supply_demand_data.json
   ✓ Data copied to C:\Users\melaea\LNGFeedGas\supply_demand_data.json

[3/5] Generating dashboard HTML...
   Generating dashboard HTML...
   ✓ Dashboard saved to C:\Users\m